### Chunking ###
- Su dung ky thuat sematic chunking de tach ca doan van thanh cac chunk nho hon
- Su dung model embeding cua AITeamVN de embedding cac chunk

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding")
document_path = "/home/quang/Downloads/Kinh_te_cong_nghiep.pdf"

/home/quang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = PyPDFLoader(document_path)
documents = loader.load()

In [4]:
raw_texts = [doc.page_content for doc in documents]
chunker = SemanticChunker(embeddings=embeddings,
                            breakpoint_threshold_type="percentile",
                            breakpoint_threshold_amount=80)
chunks = chunker.create_documents(raw_texts)

for chunk in chunks:
    chunk.page_content = chunk.page_content.replace("\n", " ")
    chunk.page_content = " ".join(chunk.page_content.split())

In [5]:
print(f"Number of chunks: {len(chunks)}")
print(f"First chunk: {chunks[0].page_content}")

Number of chunks: 114
First chunk: 1 ĐẠI HỌC THÁI NGUYÊN TRƯỜNG ĐẠI HỌC KỸ THUẬT CÔNG NGHIỆP CHƯƠNG TRÌNH ĐÀO TẠO TỪ XA TRÌNH ĐỘ ĐẠI HỌC NGÀNH: KINH TẾ CÔNG NGHIỆP CHUYÊN NGÀNH: KẾ TOÁN DOANH NGHIỆP CÔNG NGHIỆP


### Vector DB ###
- Su dung FAISS la vector store luu tru cac vector embedding

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [7]:
vector_db = None

def create_vector_store(chunks: Document) -> FAISS:
    vector_db = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    return vector_db
def save_vector_store(path: str) -> None:
    """
    Save the vector store to the specified path.
    """
    if vector_db is not None:
        vector_db.save_local(path)

def load_vector_store(path: str) -> None:
    """
    Load the vector store from the specified path.
    """
    vector_db = FAISS.load_local(
        path, 
        embeddings, 
        allow_dangerous_deserialization=True
    )

In [8]:
vector_db = create_vector_store(chunks)

In [9]:
print(f"Vector store type: {type(vector_db)}")

Vector store type: <class 'langchain_community.vectorstores.faiss.FAISS'>


### rerank ###
- Su dung mode BAAI/bge-rerank de lay cac doan van co diem top k cao

In [10]:
from sentence_transformers import CrossEncoder

In [11]:
re_rank = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cpu")

In [12]:
def rerank(self, query : str, passages : list[str]) -> tuple[list[float], list[str]]:
    # Gop query voi chunks 
    query_passages_pairs = ([query, passage] for passage in passages)
    
    # Lay diem cua doan van
    scores = re_rank.predict(list(query_passages_pairs))
    
    # Sort doan van theo diem
    rank_passages = [passage for _, passage in sorted(zip(scores, passages), key=lambda x : x[0], reverse=True)] 
    rank_scores = sorted(scores, reverse=True)
    
    return rank_scores, rank_passages

### LLM Define ###
- Dinh nghia mo hinh LLM su dung 

In [13]:
from dotenv import load_dotenv
import os
load_dotenv()
base_url = os.getenv("BASE_URL")
api_key = os.getenv("API_KEY")

In [14]:
from openai import OpenAI

LLM = OpenAI(base_url=base_url, api_key=api_key,)

In [15]:
def generate_content(query: str):
    completion = LLM.chat.completions.create(
        model="openai/gpt-oss-20b:fireworks-ai",
        messages=query,
    )
    return completion.choices[0].message.content

### Retrieval ###
- Truy xuat ra cac vector embedding co do tuong dong voi query
- Thiet ke prompt engeering
- Lay ket qua tra ve

In [16]:
query = "Kinh tế công nghiệp là gì?"

In [17]:
query_embedding = embeddings.embed_query(query)
print("len", len(query_embedding))
retrieved = vector_db.similarity_search_by_vector(
    embedding=query_embedding,
    k=5
)
print(f"Retrieved {len(retrieved)} passages.")
print(f"Retrieved passages: {retrieved}")

passages = [r.page_content for r in retrieved]
print(f"Retrieved passages: {passages}")


len 1024
Retrieved 5 passages.
Retrieved passages: [Document(id='ff0cef0f-443f-4a5d-b4de-1ccc56e162c0', metadata={}, page_content='42 14.3. Định hướng nghề nghiệp ngành Kinh tế công nghiệp Học phần Định hướng nghề nghiệp ngành Kinh tế công nghiệp là học phần tự chọn dành cho sinh viên khối ngành kinh tế. Học phần này bao gồm các nội dung: giới thiệu chương trình đào tạo (mục tiêu, chuẩn đầu ra, khung chương trình), giới thiệu ngành nghề (các vị trí việc làm và khung năng lực gắn với vị trí việc làm), hướng dẫn xây dựng mục tiêu học tập, rèn luyện và lập kế hoạch phát triển bản thân.'), Document(id='73aaa67f-3ed2-4b5e-9b29-daa2a7bffa4f', metadata={}, page_content='12. Tin học trong Kinh tê công nghiệp Học phần Tin học trong Kinh tế công nghiệp là học phần bắt buộc dành cho sinh viên ngành Kinh tế công nghiệp, bao gồm các nội dung: Tin học văn phòng; các phần mềm ứng dụng trong tài chính, kế toán trong doanh nghiệp công nghiệp; soạn thảo văn bản hợp đồng kinh tế. Học phần này sẽ giúp sin

In [18]:
combined_prompt = (
    f"Hãy trở thành chuyên gia tư vấn tuyển sinh đa ngành nghề của trường Đại học Kỹ thuật Công nghiệp - Đại học Thái Nguyên.\n"
    f"Câu hỏi của khách hàng: {query}\n"
    f"Dựa vào các thông tin sau, hãy trả lời:\n{passages}\n"
)
data = [
        {"role": "user", "content": combined_prompt}
    ]

# Step 6: Gọi LLM
response = generate_content(query=data)
print("Response:", response)

Response: **Kinh tế công nghiệp là gì?**

> **Kinh tế công nghiệp** — đánh dấu giao thoa giữa **nhật thực kinh tế** (cung‑đầu, giá cả, chuỗi giá trị) và **công nghiệp** (sản xuất, công nghệ, doanh nghiệp lớn).  
> Đây là lĩnh vực nghiên cứu cách thức các doanh nghiệp công nghiệp vận hành, tối ưu hoá chi phí, quản lý tài chính, phát triển công nghệ và đưa ra các quyết định chiến lược nhằm đạt hiệu quả cạnh tranh và đóng góp vào phát triển kinh tế quốc dân.

---

### 1. Nội dung và phương pháp nghiên cứu

| Khía cạnh | Nội dung chính |
|-----------|----------------|
| **Dấu hiệu kinh tế** | Phân tích cầu, cung, giá trị sản xuất; đánh giá tính hiệu quả tài nguyên (tài chính, lao động, công nghệ). |
| **Hệ thống sản xuất** | Kiến trúc dây chuyền, chuỗi cung ứng, quản lí tồn kho, công nghệ tự động hoá. |
| **Kỳ vọng thị trường** | Nhu cầu tiêu dùng, xu hướng sản phẩm, công nghệ mới và tác động của chính sách công. |
| **Quản lý tài chính** | Kế toán doanh nghiệp công nghiệp, chi phí sản xuấ